# [LAB-04] 10. 더미변수

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
from jussam import load_data

# 데이터프레임, 더미변수 생성 함수, 데이터 병합 함수
from pandas import DataFrame, get_dummies, merge

# 원-핫 인코딩 클래스 (머신러닝용 더미변수로 이해하세요)
from sklearn.preprocessing import OneHotEncoder

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 데이터 불러오기

In [2]:
origin = load_data('nursing_grades')
origin

📚 어느 간호학과 대학원에 지원한 학생들에 대한 합격/불합격 여부를 조사한 가상의 데이터(메타데이터 없음)


,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부
0,NRS0001,장은우,남,380,3.610,3,불합격
1,NRS0002,최지호,남,660,3.670,3,합격
2,NRS0003,김하준,남,800,4.000,1,합격
3,NRS0004,임아윤,여,640,3.190,4,합격
4,NRS0005,강하준,남,520,2.930,4,불합격
...,...,...,...,...,...,...,...
395,NRS0396,박지유,여,620,4.000,2,불합격
396,NRS0397,조하은,여,560,3.040,3,불합격
397,NRS0398,박하윤,여,460,2.630,2,불합격
398,NRS0399,이지우,여,700,3.650,2,불합격


### 3. 범주형 변수 타입 변환

- 타입 변환이 필수는 아니지만 전체 분석 과정에서는 가급적 수행하는 것이 좋다.
  - 기술통 계량 산출에 이점이 있기 때문

In [3]:
타입변환_df = origin.astype({'성별': 'category', '합격여부': 'category'})
타입변환_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   접수코드    400 non-null    object  
 1   이름      400 non-null    object  
 2   성별      400 non-null    category
 3   필기점수    400 non-null    int64   
 4   학부성적    400 non-null    float64 
 5   병원경력    400 non-null    int64   
 6   합격여부    400 non-null    category
dtypes: category(2), float64(1), int64(2), object(2)
memory usage: 16.8+ KB


## #02. 데이터 라벨링

### 1. 성별에 대한 값 종류 확인

- 명목형 값의 오름차순으로 정렬

In [4]:
vcounts = 타입변환_df['성별'].value_counts()
values = vcounts.sort_index()
values

성별
남    194
여    206
Name: count, dtype: int64

### 2. 성별에 대한 라벨링용 딕셔너리 생성

- values의 index를 라벨링에 적합한 형태로 변환
  - `{"기존값": 변환될값, "기존값": 변환될 값, ...}` 형식의 딕셔너리 구조
- **values의 인덱스는 '성별'의 고유값**
  - enumerate를 사용하여 인덱스와 값을 가져와서 labels 딕셔너리에 저장

In [5]:
labels = {}

for i, v in enumerate(values.index):
    labels[v] = i

labels

{'남': 0, '여': 1}

### 3. 성별 변수에 라벨링 적용

- 준비된 딕셔너리를 컬럼 객체(Series)의 map() 메서드에 파라미터로 전달한다.

In [6]:
라벨링_df = 타입변환_df.copy()
라벨링_df['성별'] = 라벨링_df['성별'].map(labels)
라벨링_df['성별'].value_counts().sort_index()

성별
0    194
1    206
Name: count, dtype: int64

### 4. **'합격여부'** 변수에 대한 라벨링 일괄 처리

- 합격여부는 합격,불합격,보류 세 개의 값으로 구분되어 있음

In [7]:
vcounts = 라벨링_df['합격여부'].value_counts()
values = vcounts.sort_index()

labels = {}

for i, v in enumerate(values.index):
    labels[v] = i

라벨링_df['합격여부'] = 라벨링_df['합격여부'].map(labels)
라벨링_df['합격여부'].value_counts().sort_index()

합격여부
0     23
1    250
2    127
Name: count, dtype: int64

### 5. 라벨링이 적용된 데이터 프레임 확인

- 각 변수에 대해 라벨링 된 값이 어떤 의미인지 따로 정리해 두어야 한다.

In [8]:
라벨링_df

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부
0,NRS0001,장은우,0,380,3.610,3,1
1,NRS0002,최지호,0,660,3.670,3,2
2,NRS0003,김하준,0,800,4.000,1,2
3,NRS0004,임아윤,1,640,3.190,4,2
4,NRS0005,강하준,0,520,2.930,4,1
...,...,...,...,...,...,...,...
395,NRS0396,박지유,1,620,4.000,2,1
396,NRS0397,조하은,1,560,3.040,3,1
397,NRS0398,박하윤,1,460,2.630,2,1
398,NRS0399,이지우,1,700,3.650,2,1


## #03. Pandas를 사용하는 더미변수 생성 방법

### 1. 모든 필드를 더미변수로 변환 (`N개` 생성)

- 원본 데이터프레임과 더미변수로 변환할 컬럼 이름을 파라미터로 전달한다.
- 원본 데이터프레임에서 기존의 컬럼은 제거되고 더미변수로 변경된 컬럼들이 추가된 형태의 새로운 데이터프레임이 리턴된다.
- `dtype='int'` 파라미터를 명시하지 않을 경우 `bool` 형식으로 값이 생성된다.

In [9]:
더미변수_df_1 = get_dummies(라벨링_df, columns=['성별'], dtype='int')
더미변수_df_1.head()

,접수코드,이름,필기점수,학부성적,병원경력,합격여부,성별_0,성별_1
0,NRS0001,장은우,380,3.610,3,1,1,0
1,NRS0002,최지호,660,3.670,3,2,1,0
2,NRS0003,김하준,800,4.000,1,2,1,0
3,NRS0004,임아윤,640,3.190,4,2,0,1
4,NRS0005,강하준,520,2.930,4,1,1,0


### 2. 두 개 이상의 명목형 변수 처리

- 컬럼 이름을 리스트로 묶어서 지정한다.

In [10]:
더미변수_df_2 = get_dummies(라벨링_df, columns=['성별', '합격여부'], 
                        dtype='int',)
더미변수_df_2.head()

,접수코드,이름,필기점수,학부성적,병원경력,성별_0,성별_1,합격여부_0,합격여부_1,합격여부_2
0,NRS0001,장은우,380,3.610,3,1,0,0,1,0
1,NRS0002,최지호,660,3.670,3,1,0,0,0,1
2,NRS0003,김하준,800,4.000,1,1,0,0,0,1
3,NRS0004,임아윤,640,3.190,4,0,1,0,0,1
4,NRS0005,강하준,520,2.930,4,1,0,0,1,0


### 3. `N-1`개의 더미변수 생성

- `drop_first` 파라미터를 `True`로 설정한다. (기본값=`False`)

In [11]:
더미변수_df_3 = get_dummies(라벨링_df, columns=['성별', '합격여부'], 
                          dtype='int', drop_first=True)
더미변수_df_3.head()

,접수코드,이름,필기점수,학부성적,병원경력,성별_1,합격여부_1,합격여부_2
0,NRS0001,장은우,380,3.610,3,0,1,0
1,NRS0002,최지호,660,3.670,3,0,0,1
2,NRS0003,김하준,800,4.000,1,0,0,1
3,NRS0004,임아윤,640,3.190,4,1,0,1
4,NRS0005,강하준,520,2.930,4,0,1,0


## #04. Scikit-Learn을 사용하여 단일 컬럼의 더미변수 생성

### 1. 인코딩 대상 변수 추출

In [12]:
X = 타입변환_df[['성별']]
X.head()

,성별
0,남
1,남
2,남
3,여
4,남


### 2. OneHotEncoder 처리

- **drop=None**: 첫 번째 항목 삭제 안함
- **drop="first"**: 첫 번째 항목 삭제함
- **dtype=int**: 인코딩 결과가 정수로 생성됨(기본값 float 타입)

In [13]:
encoder = OneHotEncoder(sparse_output=False, drop=None, dtype=int)
result = encoder.fit_transform(X)
# 출력 량이 많으므로 10개만 출력함
result[:10]

array([[1, 0],
       [1, 0],
       [1, 0],
       [0, 1],
       [1, 0],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [1, 0]])

### 3. 인코딩 결과를 데이터 프레임으로 생성

In [14]:
# 컬럼명 생성
new_cols = encoder.get_feature_names_out(['성별'])
new_cols

array(['성별_남', '성별_여'], dtype=object)

In [15]:
# DataFrame 병합
one_hot_df = DataFrame(result, columns=new_cols, index=타입변환_df.index)
one_hot_df.head()

,성별_남,성별_여
0,1,0
1,1,0
2,1,0
3,0,1
4,1,0


### 4. 원본 데이터 프레임과 병합

- 원본 컬럼(성별)이 그대로 남아 있기 때문에 drop() 함수를 사용해서 삭제할 필요가 있다.

In [16]:
인코딩_df_1 = merge(타입변환_df, one_hot_df, 
                   left_index=True, right_index=True)
인코딩_df_1.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,성별_남,성별_여
0,NRS0001,장은우,남,380,3.610,3,불합격,1,0
1,NRS0002,최지호,남,660,3.670,3,합격,1,0
2,NRS0003,김하준,남,800,4.000,1,합격,1,0
3,NRS0004,임아윤,여,640,3.190,4,합격,0,1
4,NRS0005,강하준,남,520,2.930,4,불합격,1,0


### 5. 불필요한 컬럼 제거

In [17]:
인코딩_df_1_최종 = 인코딩_df_1.drop(columns=['성별'])
인코딩_df_1_최종.head()

,접수코드,이름,필기점수,학부성적,병원경력,합격여부,성별_남,성별_여
0,NRS0001,장은우,380,3.610,3,불합격,1,0
1,NRS0002,최지호,660,3.670,3,합격,1,0
2,NRS0003,김하준,800,4.000,1,합격,1,0
3,NRS0004,임아윤,640,3.190,4,합격,0,1
4,NRS0005,강하준,520,2.930,4,불합격,1,0


## #05. Scikit-Learn을 사용하여 여러 컬럼의 더미변수 생성

### 1. 여러 컬럼에 대한 일괄 처리

In [18]:
X = 타입변환_df[['성별', '합격여부']]

encoder = OneHotEncoder(sparse_output=False, drop=None, dtype='int')
result = encoder.fit_transform(X)

result

array([[1, 0, 0, 1, 0],
       [1, 0, 0, 0, 1],
       [1, 0, 0, 0, 1],
       ...,
       [0, 1, 0, 1, 0],
       [0, 1, 0, 1, 0],
       [0, 1, 0, 1, 0]], shape=(400, 5))

### 2. 컬럼명 생성

In [19]:
new_cols = encoder.get_feature_names_out(['성별', '합격여부'])
new_cols

array(['성별_남', '성별_여', '합격여부_보류', '합격여부_불합격', '합격여부_합격'], dtype=object)

### 3. 데이터 프레임으로 병합

In [20]:
one_hot_df = DataFrame(result, columns=new_cols, index=타입변환_df.index)
one_hot_df.head()

,성별_남,성별_여,합격여부_보류,합격여부_불합격,합격여부_합격
0,1,0,0,1,0
1,1,0,0,0,1
2,1,0,0,0,1
3,0,1,0,0,1
4,1,0,0,1,0


### 4. 원본 데이터 프레임과 병합

In [21]:
인코딩_df_2 = merge(타입변환_df, one_hot_df, 
                   left_index=True, right_index=True)
인코딩_df_2.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,성별_남,성별_여,합격여부_보류,합격여부_불합격,합격여부_합격
0,NRS0001,장은우,남,380,3.610,3,불합격,1,0,0,1,0
1,NRS0002,최지호,남,660,3.670,3,합격,1,0,0,0,1
2,NRS0003,김하준,남,800,4.000,1,합격,1,0,0,0,1
3,NRS0004,임아윤,여,640,3.190,4,합격,0,1,0,0,1
4,NRS0005,강하준,남,520,2.930,4,불합격,1,0,0,1,0


### 5. 불필요한 컬럼 제거

In [22]:
인코딩_df_2.drop(columns=['성별', '합격여부'], inplace=True)
인코딩_df_2.head()

,접수코드,이름,필기점수,학부성적,병원경력,성별_남,성별_여,합격여부_보류,합격여부_불합격,합격여부_합격
0,NRS0001,장은우,380,3.610,3,1,0,0,1,0
1,NRS0002,최지호,660,3.670,3,1,0,0,0,1
2,NRS0003,김하준,800,4.000,1,1,0,0,0,1
3,NRS0004,임아윤,640,3.190,4,0,1,0,0,1
4,NRS0005,강하준,520,2.930,4,1,0,0,1,0
